# 00 — 공통 3-action outcome matrix

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

기존 통합 데이터의 split을 그대로 사용하면서 RouterBench와 xRouteBench의 train-only Pareto frontier에서 `low/medium/high` 대표 arm을 고릅니다. test는 파일로 전달되지만 이 노트북에서 점수를 열거나 선택에 사용하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive = '/content/drive/MyDrive/LLM_Router/colab-inputs/routing-objective-colab-code.tar.gz'


In [ ]:
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install --no-deps -e ml
import json, os, pathlib, shutil


In [ ]:
BASE_DRIVE = pathlib.Path('/content/drive/MyDrive/LLM_Router/router-model-comparison/router-model-comparison-v1/dataset')
if not (BASE_DRIVE / '_SUCCESS').is_file():
    raise RuntimeError('기존 00_prepare_dataset_and_thresholds.ipynb를 먼저 완료하세요.')
BASE_LOCAL = pathlib.Path('/content/LLM_Router/ml/artifacts/router-model-comparison-v1/dataset')
if BASE_LOCAL.exists(): shutil.rmtree(BASE_LOCAL)
BASE_LOCAL.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(BASE_DRIVE, BASE_LOCAL)
print('base dataset 복원:', BASE_LOCAL)


In [ ]:
# 원천 revision과 checksum은 기존 config가 검증합니다.
!routerbench-labeling --config ml/config/routerbench_labeling_v3.json download --data-dir ml/data/raw
!routerbench-labeling --config ml/config/routerbench_labeling_v3.json label --data-dir ml/data/raw --trust-pickle
!routerbench-labeling --config ml/config/routerbench_labeling_v3.json preprocess --profile full --seed 42
!routing-comparison-data --config ml/config/router_model_comparison_v1.json download-xroutebench


In [ ]:
LOCAL = pathlib.Path('/content/routing-objective-comparison-v1/dataset')
!routing-objective-comparison --config ml/config/routing_objective_comparison_v1.json build-data --project-root /content/LLM_Router --output-dir {LOCAL}
manifest = json.loads((LOCAL / 'manifest.json').read_text())
arms = json.loads((LOCAL / 'arms.json').read_text())
print(json.dumps(manifest['row_counts'], indent=2))
print(json.dumps(arms, indent=2, ensure_ascii=False))


In [ ]:
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/LLM_Router/routing-objective-comparison/v1')
staging = DRIVE_ROOT / 'dataset.staging'
final = DRIVE_ROOT / 'dataset'
if staging.exists(): shutil.rmtree(staging)
staging.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL, staging)
if final.exists():
    old = json.loads((final / 'manifest.json').read_text())
    new = json.loads((staging / 'manifest.json').read_text())
    if old['artifacts'] != new['artifacts'] or old['arms'] != new['arms']:
        raise RuntimeError('같은 dataset version에 다른 outcome matrix가 존재합니다.')
    shutil.rmtree(staging)
else:
    os.replace(staging, final)
print('봉인 완료:', final)
